In [ ]:
import torch
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from concept_erasure import LeaceEraser

# 1. Create synthetic data: 2048 samples, 128 features, 2 classes
n, d, k = 2048, 128, 2
X, Y = make_classification(
    n_samples=n,
    n_features=d,
    n_classes=k,
    random_state=42,
)

X_t = torch.from_numpy(X)
Y_t = torch.from_numpy(Y)

# 2. Baseline: a logistic regression classifier CAN learn the concept (Y) from X
real_lr = LogisticRegression(max_iter=1000).fit(X, Y)
beta_before = torch.from_numpy(real_lr.coef_)

print("--- Before Erasure ---")
print(f"Norm of learned coefficients: {beta_before.norm(p=torch.inf):.4f}")
assert beta_before.norm(p=torch.inf) > 0.1, "Baseline classifier should learn something."

# 3. Apply LEACE: fit an eraser to remove the concept Y from X
eraser = LeaceEraser.fit(X_t, Y_t)
X_erased = eraser(X_t)

# 4. Verification: a new classifier should FAIL to learn the concept from erased data
null_lr = LogisticRegression(max_iter=1000, tol=0.0).fit(X_erased.numpy(), Y)
beta_after = torch.from_numpy(null_lr.coef_)

print("\n--- After Erasure ---")
print(f"Norm of learned coefficients: {beta_after.norm(p=torch.inf):.6f}")
assert beta_after.norm(p=torch.inf) < 1e-4, "LEACE should have removed the linear signal."

print("\n✅ Success: LEACE correctly erased the concept.")

In [ ]:
import torch
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from concept_erasure import LeaceEraser

# 1. Create features X and TWO separate concept labels Y1, Y2
n, d = 2048, 128
X, Y1 = make_classification(n_samples=n, n_features=d, n_classes=2, random_state=42)
# Create a second, independent concept
_, Y2 = make_classification(n_samples=n, n_features=d, n_classes=2, random_state=123)

X_t = torch.from_numpy(X).float()
Y1_t = torch.from_numpy(Y1).float()
Y2_t = torch.from_numpy(Y2).float()

# 2. Stack the concepts into a single multi-column tensor Z
Z_t = torch.stack([Y1_t, Y2_t], dim=1)  # shape: (n, 2)

# 3. Baseline: classifiers can learn each concept from raw X
lr1 = LogisticRegression(max_iter=1000).fit(X, Y1)
lr2 = LogisticRegression(max_iter=1000).fit(X, Y2)
print(f"Before erasure, |coef1| = {np.linalg.norm(lr1.coef_, ord=np.inf):.4f}")
print(f"Before erasure, |coef2| = {np.linalg.norm(lr2.coef_, ord=np.inf):.4f}")

# 4. Fit LEACE with BOTH concepts at once
eraser = LeaceEraser.fit(X_t, Z_t)
X_erased = eraser(X_t).numpy()

# 5. Verification: classifiers should now fail to learn either concept
null_lr1 = LogisticRegression(max_iter=1000, tol=0.0).fit(X_erased, Y1)
null_lr2 = LogisticRegression(max_iter=1000, tol=0.0).fit(X_erased, Y2)
print(f"\nAfter erasure, |coef1| = {np.linalg.norm(null_lr1.coef_, ord=np.inf):.6f}")
print(f"After erasure, |coef2| = {np.linalg.norm(null_lr2.coef_, ord=np.inf):.6f}")

assert np.linalg.norm(null_lr1.coef_, ord=np.inf) < 1e-4
assert np.linalg.norm(null_lr2.coef_, ord=np.inf) < 1e-4
print("\n✅ Success: Joint erasure removed linear information for both concepts.")

In [ ]:
%load_ext autoreload
%autoreload 2

import umap
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from trace_comparison.triplet_accuracy import triplet_accuracy, relatedness_levels
from trace_comparison.data.dataset import load_trace_dataset
from utils.notebook_utils import reset_cwd_and_env

reset_cwd_and_env()
base_path = Path(os.path.join(os.getcwd()))

In [ ]:
dataset = load_trace_dataset(
    base_path / "data" / "traces_dataset" / "nonograms" / "base_1000.jsonl")

In [ ]:
print(len(dataset.traces))